In [1]:
import warnings
from rdkit import RDLogger

# 屏蔽 RDKit 警告
RDLogger.DisableLog('rdApp.*')

# 或屏蔽所有 Python 警告
warnings.filterwarnings("ignore")
# 屏蔽 LightGBM 警告
warnings.filterwarnings("ignore", category=UserWarning, module="lightgbm")

In [2]:
import torch
from sklearn.model_selection import StratifiedKFold
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.metrics import precision_recall_curve, auc
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import joblib
import optuna
from rdkit.Chem import Descriptors, AllChem
from tqdm import tqdm  # 导入tqdm
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GroupKFold





In [3]:
# 函数：将SMILES转换为分子描述符和指纹
def smiles_to_features(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    # 提取描述符
    descriptors = [
        Descriptors.MolWt(mol),  # 分子量
        Descriptors.MolLogP(mol),  # LogP
        Descriptors.NumHDonors(mol),  # 氢键供体数量
        Descriptors.NumHAcceptors(mol)  # 氢键受体数量
    ]
    # 生成Morgan指纹
    fingerprint = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048)
    fingerprint_array = np.zeros((2048,))
    Chem.DataStructs.ConvertToNumpyArray(fingerprint, fingerprint_array)
    # 合并描述符和指纹
    features = np.concatenate([descriptors, fingerprint_array])
    return features


In [4]:
def train_evaluate_regression_model_with_optuna(model_name, model_class, param_func, X, y, groups):
    def objective(trial):
        params = param_func(trial)
        model = model_class(**params)

        gkf = GroupKFold(n_splits=10)  # 改为十折交叉验证
        errors = []

        for train_idx, val_idx in tqdm(gkf.split(X, y, groups=groups), total=10, desc=f"Training {model_name}"):
            X_train, X_val = X[train_idx], X[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]

            model.fit(X_train, y_train)
            y_pred = model.predict(X_val)

            # 计算错误，较大值为分子，较小值为分母
            error = np.maximum(y_val, y_pred) / np.minimum(y_val, y_pred)
            errors.append(np.median(error))  # 中值误差

        return np.mean(errors)

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=30)

    print(f'Best parameters for {model_name}: {study.best_params}')
    

In [5]:
# 数据预处理
df = pd.read_excel('../fish_unique.xlsx')
labels = df['mgperL'].values
smiles_list = df['SMILES_Canonical_RDKit'].tolist()
endpoints_a = df['endpoint']
Duration_Values_a = df['Duration_Value'].values
effects_a = df['effect']


In [6]:

features = []
new_labels = []
new_smiles_list = []
endpoints = []
Duration_Values = []
effects =[]


for smiles, label,a,b,c in zip(smiles_list, labels,Duration_Values_a,effects_a,endpoints_a):
    feature = smiles_to_features(smiles)
    if feature is not None:
        features.append(feature)
        new_labels.append(label)
        new_smiles_list.append(smiles)
        Duration_Values.append(a)
        effects.append(b)
        endpoints.append(c)

X = np.array(features)
y = np.array(new_labels)
groups = new_smiles_list  # 可直接用于 GroupKFold




In [7]:
def encode_column(zz):
    zz_series = pd.Series(zz)  # 转换为 Series
    unique_values = zz_series.unique()
    if len(unique_values) > 1:
        encoder = OneHotEncoder(sparse_output=False)
        return encoder.fit_transform(zz_series.values.reshape(-1, 1))
    else:
        return None  # 只有一种类别时忽略

Duration_Values =pd.Series(Duration_Values)


# 编码 effect、endpoint 和 species_group 列
effect_encoded = encode_column(effects)
endpoint_encoded = encode_column(endpoints)
#species_encoded = encode_column(df, 'species_group')

# # 将需要的列拼接成输入 X
X = np.hstack((X, Duration_Values.values.reshape(-1, 1)))

# # 拼接编码后的列（如果存在）
for encoded_feature in [effect_encoded, endpoint_encoded]:
     if encoded_feature is not None:
         X = np.hstack((X, encoded_feature))



y=np.log1p(y)

In [8]:
import os
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.model_selection import GroupKFold
from tqdm import tqdm

# 1. 最优参数
best_params_xgb =  {
    'n_estimators': 568, 
    'max_depth': 20, 
    'learning_rate': 0.0548419024359913, 
    'subsample': 0.6199824702604411, 
    'colsample_bytree': 0.6669061263322058, 
    'reg_alpha': 0.928075881586975, 
    'reg_lambda': 0.9869950065551683
}


# 2. 定义保存文件夹
save_dir = './xgb_cv_results'
os.makedirs(save_dir, exist_ok=True)

# 3. 十折交叉验证
gkf = GroupKFold(n_splits=10)

for fold, (train_idx, val_idx) in enumerate(tqdm(gkf.split(X, y, groups=groups), total=10, desc="XGBoost Cross Validation")):
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    # 构建 XGBoost 模型
    model = XGBRegressor(**best_params_xgb, random_state=42, n_jobs=-1, verbosity=0)
    model.fit(X_train, y_train)

    # 验证集预测
    y_pred = model.predict(X_val)

    # 4. 保存每一折的真实值和预测值
    np.save(os.path.join(save_dir, f'fold_{fold}_y_true.npy'), y_val)
    np.save(os.path.join(save_dir, f'fold_{fold}_y_pred.npy'), y_pred)

print(f"XGBoost十折交叉验证完成，结果已保存到 {save_dir} 文件夹。")


XGBoost Cross Validation: 100%|██████████| 10/10 [18:09<00:00, 108.95s/it]

XGBoost十折交叉验证完成，结果已保存到 ./xgb_cv_results 文件夹。


In [9]:
import os
import numpy as np
from lightgbm import LGBMRegressor
from sklearn.model_selection import GroupKFold
from tqdm import tqdm

# 1. 最优参数
best_params_lgbm = {'n_estimators': 353, 'max_depth': 20, 'num_leaves': 170, 'learning_rate': 0.11212245188752953, 'feature_fraction': 0.7673505068979276, 'bagging_fraction': 0.9737113542482807, 'bagging_freq': 4, 'reg_alpha': 0.6115055127735106, 'reg_lambda': 0.4867933887145545,
    'verbose': -1
}

# 2. 定义保存文件夹
save_dir = './lgbm_cv_results'
os.makedirs(save_dir, exist_ok=True)

# 3. 十折交叉验证
gkf = GroupKFold(n_splits=10)

for fold, (train_idx, val_idx) in enumerate(tqdm(gkf.split(X, y, groups=groups), total=10, desc="LightGBM Cross Validation")):
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    # 构建 LightGBM 模型
    model = LGBMRegressor(**best_params_lgbm,objective="poisson", random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)
    # 验证集预测
    y_pred = model.predict(X_val)

    # 4. 保存每一折的真实值和预测值
    np.save(os.path.join(save_dir, f'fold_{fold}_y_true.npy'), y_val)
    np.save(os.path.join(save_dir, f'fold_{fold}_y_pred.npy'), y_pred)

print(f"LightGBM十折交叉验证完成，结果已保存到 {save_dir} 文件夹。")

LightGBM Cross Validation: 100%|██████████| 10/10 [01:21<00:00,  8.16s/it]

LightGBM十折交叉验证完成，结果已保存到 ./lgbm_cv_results 文件夹。


In [10]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from tqdm import tqdm
import random

# 固定随机种子
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # for multi-GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)  # 设置固定种子




# 最优超参数
best_params_dnn = {
    'hidden_layer_sizes': (150,),
    'activation': 'relu',
    'alpha': 7.490484670539513e-05,
    'learning_rate': 0.009345382511909396,
    'optimizer': 'sgd'
}

# 结果保存路径
save_dir = './dnn_cv_results'
os.makedirs(save_dir, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 自定义网络结构
class DNNWithSoftplus(nn.Module):
    def __init__(self, input_dim, hidden_sizes, activation):
        super().__init__()
        act_fn = {
            'relu': nn.ReLU(),
            'logistic': nn.Sigmoid(),
            'tanh': nn.Tanh()
        }[activation]
        layers = []
        prev_dim = input_dim
        for h in hidden_sizes:
            layers += [nn.Linear(prev_dim, h), act_fn]
            prev_dim = h
        layers += [nn.Linear(prev_dim, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return F.softplus(self.net(x)).squeeze(-1)

# 十折交叉验证
gkf = GroupKFold(n_splits=10)

for fold, (train_idx, val_idx) in enumerate(tqdm(gkf.split(X, y, groups=groups), total=10, desc="PyTorch DNN CV")):
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)

    model = DNNWithSoftplus(
        input_dim=X.shape[1],
        hidden_sizes=best_params_dnn['hidden_layer_sizes'],
        activation=best_params_dnn['activation']
    ).to(device)

    optimizer = {
        'adam': torch.optim.Adam,
        'sgd': torch.optim.SGD
    }[best_params_dnn['optimizer']](
        model.parameters(),
        lr=best_params_dnn['learning_rate'],
        weight_decay=best_params_dnn['alpha']
    )

    loss_fn = nn.MSELoss()
    train_ds = TensorDataset(torch.tensor(X_train).float(), torch.tensor(y_train).float())
    train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)

    model.train()
    for epoch in range(200):  # 保持与 MLPRegressor 一致的 200 epoch
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            preds = model(xb)
            loss = loss_fn(preds, yb)
            loss.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        X_val_tensor = torch.tensor(X_val).float().to(device)
        y_pred = model(X_val_tensor).cpu().numpy()

    # 保存预测结果
    np.save(os.path.join(save_dir, f'fold_{fold}_y_true.npy'), y_val)
    np.save(os.path.join(save_dir, f'fold_{fold}_y_pred.npy'), y_pred)

print(f"✅ PyTorch DNN 十折交叉验证完成，结果已保存到：{save_dir}")


PyTorch DNN CV: 100%|██████████| 10/10 [09:53<00:00, 59.35s/it]

✅ PyTorch DNN 十折交叉验证完成，结果已保存到：./dnn_cv_results
